# CJK-TokenLab: Final Research Workflow (Local)
This notebook executes the entire 5,000-sentence experiment locally on Windows.


In [ ]:
# ============================================================
# 1. LOCAL ENVIRONMENT SETUP
# ============================================================
import os
import sys
from pathlib import Path

# Current CJK-TokenLab project
PROJECT_DIR = Path.cwd().resolve()

print("=" * 60)
print("CJK-TokenLab LOCAL ENVIRONMENT")
print("=" * 60)

print("Project directory:", PROJECT_DIR)

# Make project importable
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Local artifact directories
MODELS_DIR = PROJECT_DIR / "models"
TOKENIZERS_DIR = PROJECT_DIR / "tokenizers"
RESULTS_DIR = PROJECT_DIR / "results"
SPLITS_DIR = PROJECT_DIR / "data" / "splits"
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"

for directory in [MODELS_DIR, TOKENIZERS_DIR, RESULTS_DIR, SPLITS_DIR, RAW_DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Environment variables used by the project
os.environ["MODELS_DIR"] = str(MODELS_DIR)
os.environ["TOKENIZERS_DIR"] = str(TOKENIZERS_DIR)
os.environ["RESULTS_DIR"] = str(RESULTS_DIR)
os.environ["SPLITS_DIR"] = str(SPLITS_DIR)
os.environ["RAW_DATA_DIR"] = str(RAW_DATA_DIR)

print("Local environment configured successfully.")
print("Models    :", MODELS_DIR)
print("Tokenizers:", TOKENIZERS_DIR)
print("Results   :", RESULTS_DIR)
print("Splits    :", SPLITS_DIR)
print("Raw data  :", RAW_DATA_DIR)

In [ ]:
# ============================================================
# 2. DEPENDENCY CHECK
# ============================================================
import pkg_resources

try:
    with open(PROJECT_DIR / "requirements.txt", "r", encoding="utf-8") as f:
        requirements = [line.strip() for line in f if line.strip() and not line.startswith("#")]
    pkg_resources.require(requirements)
    print("All dependencies from requirements.txt are satisfied.")
except Exception as e:
    print(f"Dependency warning/error: {e}")

In [ ]:
# ============================================================
# 3. RTX 3050 / CUDA VERIFICATION
# ============================================================
import torch
import sys

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM:", round(props.total_memory / (1024 ** 3), 2), "GB")
    device = torch.device("cuda")
else:
    print("WARNING: CUDA GPU is NOT available.")
    print("Falling back to CPU. Note: Transformer training will be slower.")
    device = torch.device("cpu")

print("Selected device:", device)

In [ ]:
# ============================================================
# 4. PROJECT STRUCTURE VERIFICATION
# ============================================================
required_items = ["config", "src", "data", "requirements.txt", "main.py"]
missing = []

for item in required_items:
    if not (PROJECT_DIR / item).exists():
        missing.append(item)

if missing:
    raise FileNotFoundError(
        "This does not appear to be the complete CJK-TokenLab project.\n"
        f"Missing: {missing}\n"
        f"Current directory: {PROJECT_DIR}"
    )

print("Project structure verified.")

In [ ]:
# ============================================================
# 5. EXECUTE SOURCE VERIFICATION
# ============================================================
execute_repo_dir = PROJECT_DIR / "data" / "execute_repo"
execute_main_dir = execute_repo_dir / "EXECUTE-main"
tiny_stories_path = execute_main_dir / "data" / "TinyStoriesV2-GPT4-valid.txt"

if not tiny_stories_path.exists():
    print("Downloading official EXECUTE repository...")
    execute_repo_dir.mkdir(parents=True, exist_ok=True)
    os.system(f"git clone https://github.com/Leukas/EXECUTE.git {execute_main_dir}")
    
if tiny_stories_path.exists():
    print(f"Verified: EXECUTE source exists at {tiny_stories_path}!")
else:
    raise Exception(f"Failed to locate TinyStories source at {tiny_stories_path}")

In [ ]:
# ============================================================
# 6. CONFIGURATION
# ============================================================
from config.config import LANGUAGES, FULL_TRAINING_PARAMS

TOTAL_SENTENCES = 1000
TRAIN_SENTENCES = 800
VAL_SENTENCES = 100
TEST_SENTENCES = 100
FINAL_VOCAB_SIZE = 2000

print("Experiment Configuration:")
print(f"Total Sentences: {TOTAL_SENTENCES}")
print(f"Train/Val/Test: {TRAIN_SENTENCES} / {VAL_SENTENCES} / {TEST_SENTENCES}")
print(f"Final Vocabulary Size: {FINAL_VOCAB_SIZE}")

In [ ]:
# ============================================================
# 7. EXISTING DATASET / RESUME CHECK
# ============================================================
import json

print("Checking for existing dataset files...")
for lang in LANGUAGES.keys():
    file_path = RAW_DATA_DIR / f"{lang}_sentences.jsonl"
    count = 0
    if file_path.exists():
        with open(file_path, 'r', encoding='utf-8') as f:
            count = sum(1 for _ in f)
    print(f"{LANGUAGES[lang]} existing translated sentences: {count}/{TOTAL_SENTENCES}")

In [ ]:
# ============================================================
# 8. GENERATE / RESUME 5000-SENTENCE DATASET
# ============================================================
from src.data.prepare_execute_dataset import prepare_execute_dataset

print(f"Generating {TOTAL_SENTENCES}-sample EXECUTE-aligned dataset...")
# Deep_translator safely resumes if rate-limited. If it stops, just rerun this cell.
prepare_execute_dataset(max_sentences=TOTAL_SENTENCES)
print("Dataset generation process finished.")

In [ ]:
# ============================================================
# 9. VERIFY 1000/1000/1000
# ============================================================
print("Verifying dataset exact sizes and integrity...")

for lang in LANGUAGES.keys():
    file_path = RAW_DATA_DIR / f"{lang}_corpus.txt"
    if not file_path.exists():
        raise FileNotFoundError(f"Dataset file missing for {lang}")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    total_lines = len(lines)
    non_empty = [line for line in lines if line.strip()]
    empty_lines = total_lines - len(non_empty)
    
    # Basic english presence check fallback for plain text
    english_only_count = sum(1 for line in non_empty if line.isascii() and any(c.isalpha() for c in line))

    print(f"\n--- {LANGUAGES[lang]} ---")
    print(f"Total lines: {total_lines}")
    print(f"Non-empty lines: {len(non_empty)}")
    print(f"Empty lines: {empty_lines}")
    print(f"Untranslated (English only): {english_only_count}")
    
    if len(non_empty) != TOTAL_SENTENCES:
        raise ValueError(f"STOP: {LANGUAGES[lang]} dataset does not have exactly {TOTAL_SENTENCES} sentences (Found {len(non_empty)}).")
        
print("\nAll datasets have exactly 1000 sentences!")

In [ ]:
# ============================================================
# 10. VERIFY TRANSLATION CONTENT
# ============================================================
print("Verifying translation content (first 3 sentences)...\n")

for lang in LANGUAGES.keys():
    file_path = RAW_DATA_DIR / f"{lang}_corpus.txt"
    print(f"=== {LANGUAGES[lang]} ===")
    with open(file_path, 'r', encoding='utf-8') as f:
        for i in range(3):
            line = f.readline()
            if line:
                print(line.strip())
    print("\n")

In [ ]:
# ============================================================
# 11. CREATE 800/100/100 SPLITS
# ============================================================
from src.preprocessing.prep import create_splits, load_split

print("Creating train/val/test splits...")
for lang_code in LANGUAGES.keys():
    print(f"\nSplitting {LANGUAGES[lang_code]}...")
    train_path, val_path, test_path = create_splits(lang_code)
    
    train_texts = load_split(lang_code, "train")
    val_texts = load_split(lang_code, "val")
    test_texts = load_split(lang_code, "test")
    
    print(f"Train : {len(train_texts)}")
    print(f"Val   : {len(val_texts)}")
    print(f"Test  : {len(test_texts)}")
    
    if len(train_texts) != TRAIN_SENTENCES or len(val_texts) != VAL_SENTENCES or len(test_texts) != TEST_SENTENCES:
        raise ValueError(
            f"STOP: Split counts are wrong for {lang_code}. "
            f"Expected {TRAIN_SENTENCES}/{VAL_SENTENCES}/{TEST_SENTENCES}"
        )
print("\nAll splits verified successfully.")

In [ ]:
# ============================================================
# 12. TRAIN TOKENIZERS
# ============================================================
from src.tokenization.baseline_tokenizers import BaselineTokenizer
from src.tokenization.cjk_tokenizer import CJKHybridTokenizer

for lang_code in LANGUAGES.keys():
    print(f"\n========================================\nTokenizers for {LANGUAGES[lang_code]}\n========================================")
    
    # Tokenizer training must use ONLY the training split
    train_texts = load_split(lang_code, "train")
    
    # Character
    print("Training Character...")
    tok_char = BaselineTokenizer(lang_code, vocab_size=FINAL_VOCAB_SIZE, method="char")
    tok_char.fit_on_texts(train_texts, str(TOKENIZERS_DIR / f"{lang_code}_char"))
    
    # Morphological
    if lang_code == 'zho':
        print("Training Jieba...")
        tok_morph = BaselineTokenizer(lang_code, vocab_size=FINAL_VOCAB_SIZE, method="jieba")
        tok_morph.fit_on_texts(train_texts, str(TOKENIZERS_DIR / f"{lang_code}_jieba"))
    elif lang_code == 'jpn':
        print("Training Nagisa...")
        tok_morph = BaselineTokenizer(lang_code, vocab_size=FINAL_VOCAB_SIZE, method="nagisa")
        tok_morph.fit_on_texts(train_texts, str(TOKENIZERS_DIR / f"{lang_code}_nagisa"))
        
    # BPE (PRIMARY BASELINE)
    print("Training BPE (Primary Baseline)...")
    tok_bpe = BaselineTokenizer(lang_code, vocab_size=FINAL_VOCAB_SIZE, method="bpe")
    tok_bpe.fit_on_texts(train_texts, str(TOKENIZERS_DIR / f"{lang_code}_bpe"))
    
    # SentencePiece
    print("Training SentencePiece...")
    tok_spm = BaselineTokenizer(lang_code, vocab_size=FINAL_VOCAB_SIZE, method="spm")
    tok_spm.fit_on_texts(train_texts, str(TOKENIZERS_DIR / f"{lang_code}_spm"))
    
    # Proposed CJK-Aware Tokenizer
    print("Training Proposed CJK-Aware Tokenizer...")
    tok_prop = CJKHybridTokenizer(lang_code, vocab_size=FINAL_VOCAB_SIZE)
    tok_prop.fit_on_texts(train_texts)
    tok_prop.save(str(TOKENIZERS_DIR / f"{lang_code}_proposed"))

In [ ]:
# ============================================================
# 13. TOKENIZATION METRICS
# ============================================================
from src.evaluation.metrics import calculate_tokenization_metrics

tokenization_metrics = {}

for lang_code in LANGUAGES.keys():
    print(f"Calculating metrics for {LANGUAGES[lang_code]}...")
    val_texts = load_split(lang_code, "val")
    
    tok_char = BaselineTokenizer.load(str(TOKENIZERS_DIR / f"{lang_code}_char"), lang_code, FINAL_VOCAB_SIZE, "char")
    tok_bpe = BaselineTokenizer.load(str(TOKENIZERS_DIR / f"{lang_code}_bpe"), lang_code, FINAL_VOCAB_SIZE, "bpe")
    tok_spm = BaselineTokenizer.load(str(TOKENIZERS_DIR / f"{lang_code}_spm"), lang_code, FINAL_VOCAB_SIZE, "spm")
    tok_prop = CJKHybridTokenizer.load(str(TOKENIZERS_DIR / f"{lang_code}_proposed"))
    
    lang_metrics = {
        "Character": calculate_tokenization_metrics(tok_char, val_texts),
        "BPE (Primary)": calculate_tokenization_metrics(tok_bpe, val_texts),
        "SentencePiece": calculate_tokenization_metrics(tok_spm, val_texts),
        "Proposed": calculate_tokenization_metrics(tok_prop, val_texts)
    }
    
    if lang_code == 'zho':
        tok_morph = BaselineTokenizer.load(str(TOKENIZERS_DIR / f"{lang_code}_jieba"), lang_code, FINAL_VOCAB_SIZE, "jieba")
        lang_metrics["Jieba"] = calculate_tokenization_metrics(tok_morph, val_texts)
    elif lang_code == 'jpn':
        tok_morph = BaselineTokenizer.load(str(TOKENIZERS_DIR / f"{lang_code}_nagisa"), lang_code, FINAL_VOCAB_SIZE, "nagisa")
        lang_metrics["Nagisa"] = calculate_tokenization_metrics(tok_morph, val_texts)
        
    tokenization_metrics[lang_code] = lang_metrics

metrics_dir = RESULTS_DIR / "tables"
metrics_dir.mkdir(parents=True, exist_ok=True)

with open(metrics_dir / "tokenization_metrics.json", "w", encoding='utf-8') as f:
    json.dump(tokenization_metrics, f, indent=4, ensure_ascii=False)

print("Tokenization metrics saved to results/tables/tokenization_metrics.json")

In [ ]:
# ============================================================
# 14. VERIFY TOKENIZERS
# ============================================================
print("Verifying saved tokenizers...\n")

for lang_code in LANGUAGES.keys():
    expected_files = [f"{lang_code}_char", f"{lang_code}_bpe", f"{lang_code}_spm", f"{lang_code}_proposed"]
    if lang_code == "zho":
        expected_files.append(f"{lang_code}_jieba")
    elif lang_code == "jpn":
        expected_files.append(f"{lang_code}_nagisa")
        
    print(f"--- {LANGUAGES[lang_code]} ---")
    for ef in expected_files:
        matches = list(TOKENIZERS_DIR.glob(f"{ef}*"))
        if not matches:
            print(f"[MISSING] {ef}")
        else:
            print(f"[FOUND] {ef}")
    print("")

In [ ]:
# ============================================================
# 15. TRAIN 6 TRANSFORMER MODELS
# ============================================================
from torch.utils.data import DataLoader
from src.training.model import SimpleTransformerLM
from src.training.train import TokenizedDataset, train_model

for lang_code in LANGUAGES.keys():
    print(f"\n========================================\nModel Training for {LANGUAGES[lang_code]}\n========================================")
    
    train_texts = load_split(lang_code, "train")
    val_texts = load_split(lang_code, "val")
    
    # Load tokenizers
    tok_bpe = BaselineTokenizer.load(str(TOKENIZERS_DIR / f"{lang_code}_bpe"), lang_code, FINAL_VOCAB_SIZE, "bpe")
    tok_prop = CJKHybridTokenizer.load(str(TOKENIZERS_DIR / f"{lang_code}_proposed"))
    
    # Get vocabulary sizes dynamically if possible, else use FINAL_VOCAB_SIZE
    vocab_size_base = tok_bpe.tokenizer.get_vocab_size() if hasattr(tok_bpe, 'tokenizer') and hasattr(tok_bpe.tokenizer, 'get_vocab_size') else FINAL_VOCAB_SIZE
    vocab_size_prop = tok_prop.tokenizer.get_vocab_size() if hasattr(tok_prop, 'tokenizer') and hasattr(tok_prop.tokenizer, 'get_vocab_size') else FINAL_VOCAB_SIZE
    
    # Baseline Model (BPE)
    print("\n--- Training Primary Baseline Model (BPE) ---")
    train_dataset_base = TokenizedDataset(train_texts, tok_bpe, FULL_TRAINING_PARAMS["max_seq_len"])
    val_dataset_base = TokenizedDataset(val_texts, tok_bpe, FULL_TRAINING_PARAMS["max_seq_len"])
    train_loader_base = DataLoader(train_dataset_base, batch_size=FULL_TRAINING_PARAMS["batch_size"], shuffle=True)
    val_loader_base = DataLoader(val_dataset_base, batch_size=FULL_TRAINING_PARAMS["batch_size"])
    
    model_base = SimpleTransformerLM(
        vocab_size=vocab_size_base,
        d_model=FULL_TRAINING_PARAMS["d_model"],
        nhead=FULL_TRAINING_PARAMS["nhead"],
        num_layers=FULL_TRAINING_PARAMS["num_layers"],
        dim_feedforward=FULL_TRAINING_PARAMS["dim_feedforward"],
        max_seq_len=FULL_TRAINING_PARAMS["max_seq_len"]
    )
    train_model(model_base, train_loader_base, val_loader_base, FULL_TRAINING_PARAMS["epochs"], FULL_TRAINING_PARAMS["lr"], device, str(MODELS_DIR / f"{lang_code}_baseline"))
    
    # Proposed Model
    print("\n--- Training Proposed Model ---")
    train_dataset_prop = TokenizedDataset(train_texts, tok_prop, FULL_TRAINING_PARAMS["max_seq_len"])
    val_dataset_prop = TokenizedDataset(val_texts, tok_prop, FULL_TRAINING_PARAMS["max_seq_len"])
    train_loader_prop = DataLoader(train_dataset_prop, batch_size=FULL_TRAINING_PARAMS["batch_size"], shuffle=True)
    val_loader_prop = DataLoader(val_dataset_prop, batch_size=FULL_TRAINING_PARAMS["batch_size"])
    
    model_prop = SimpleTransformerLM(
        vocab_size=vocab_size_prop,
        d_model=FULL_TRAINING_PARAMS["d_model"],
        nhead=FULL_TRAINING_PARAMS["nhead"],
        num_layers=FULL_TRAINING_PARAMS["num_layers"],
        dim_feedforward=FULL_TRAINING_PARAMS["dim_feedforward"],
        max_seq_len=FULL_TRAINING_PARAMS["max_seq_len"]
    )
    train_model(model_prop, train_loader_prop, val_loader_prop, FULL_TRAINING_PARAMS["epochs"], FULL_TRAINING_PARAMS["lr"], device, str(MODELS_DIR / f"{lang_code}_proposed"))

In [ ]:
# ============================================================
# 16. VERIFY MODEL CHECKPOINTS
# ============================================================
print("Verifying Transformer model checkpoints...\n")

for lang_code in LANGUAGES.keys():
    baseline_model = list(MODELS_DIR.glob(f"{lang_code}_baseline*"))
    proposed_model = list(MODELS_DIR.glob(f"{lang_code}_proposed*"))
    
    if not baseline_model:
        print(f"[MISSING] Baseline model checkpoint for {lang_code}")
    else:
        print(f"[FOUND] Baseline model for {lang_code}: {baseline_model[0].name}")
        
    if not proposed_model:
        print(f"[MISSING] Proposed model checkpoint for {lang_code}")
    else:
        print(f"[FOUND] Proposed model for {lang_code}: {proposed_model[0].name}")

In [ ]:
# ============================================================
# 17. EXECUTE EVALUATION
# ============================================================
from src.evaluation.execute_tasks import run_all_evaluations

print("\n--- Running Final EXECUTE Evaluation ---")
run_all_evaluations()
print("EXECUTE Evaluation Complete!")

In [ ]:
# ============================================================
# 18. FINAL RESULTS VERIFICATION
# ============================================================
print("Verifying final results...\n")

# tokenization_metrics.json should exist
metrics_file = RESULTS_DIR / "tables" / "tokenization_metrics.json"
if metrics_file.exists():
    print(f"[FOUND] {metrics_file.relative_to(PROJECT_DIR)}")
else:
    print(f"[MISSING] tokenization_metrics.json")

print("\nPipeline Complete! All operations ran locally on the RTX 3050.")